In [1]:
# Pregunta: ¿qué conductores tienen mayor actividad una vez que relacionamos sus métricas con el registro maestro?
# La unión permite presentar una respuesta interpretable para la operación de transporte.
#
# lectura ──► mapper de turnos ──► reducer ──► mapper de conductores ──► join por clave ──► escritura

import csv
from pathlib import Path

ROOT = next(
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "data").is_dir() and (path / "submission").is_dir()
)

TIMESHEET_SOURCE = ROOT / "data" / "timesheet.csv"
DRIVERS_SOURCE = ROOT / "data" / "drivers.csv"
MAPPER_OUTPUT = ROOT / "temp" / "driver_metric_pairs.csv"
JOIN_OUTPUT = ROOT / "temp" / "driver_activity_join.csv"
RESULT = ROOT / "submission" / "driver_activity.csv"

with TIMESHEET_SOURCE.open(encoding="utf-8", newline="") as file:
    timesheet = list(csv.DictReader(file))

with DRIVERS_SOURCE.open(encoding="utf-8", newline="") as file:
    drivers = list(csv.DictReader(file))

len(timesheet), len(drivers)

(1768, 34)

In [2]:
# Esta consulta SQL expresa la pregunta de negocio.
# La subconsulta representa la primera etapa MapReduce; después se une con el maestro de conductores.
#
# SELECT
#     drivers.name,
#     metrics.total_hours,
#     metrics.total_miles,
#     metrics.weeks
# FROM drivers
# INNER JOIN metrics
#     ON drivers.driverId = metrics.driverId
# ORDER BY metrics.total_miles DESC;

def map_pairs(records):
    pairs = []

    for record in records:
        pairs.append(
            (
                record["driverId"],
                (
                    float(record["hours-logged"]),
                    float(record["miles-logged"]),
                    1,
                ),
            )
        )

    return pairs

In [3]:
# El mapper deja explícita la clave que relacionará las dos fuentes: driverId.

mapped_pairs = map_pairs(timesheet)

with MAPPER_OUTPUT.open("w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file, lineterminator="\n")
    writer.writerow(["driverId", "hours", "miles", "weeks"])
    writer.writerows((key, *value) for key, value in mapped_pairs)

mapped_pairs[:5]

[('10', (70.0, 3300.0, 1)),
 ('10', (70.0, 3300.0, 1)),
 ('10', (60.0, 2800.0, 1)),
 ('10', (70.0, 3100.0, 1)),
 ('10', (70.0, 3200.0, 1))]

In [4]:
# El reducer agrupa las semanas de cada conductor antes de hacer la unión.

def reduce_by_key(pairs):
    grouped_values = {}

    for key, value in pairs:
        grouped_values.setdefault(key, []).append(value)

    metrics_by_driver = {}
    for driver_id, values in grouped_values.items():
        total_hours = sum(hours for hours, _, _ in values)
        total_miles = sum(miles for _, miles, _ in values)
        weeks = sum(week_count for _, _, week_count in values)
        metrics_by_driver[driver_id] = {
            "total_hours": round(total_hours, 2),
            "total_miles": round(total_miles, 2),
            "weeks": weeks,
        }

    return metrics_by_driver

In [5]:
# Esta salida ya contiene una fila por conductor, pero todavía no tiene su nombre.

metrics_by_driver = reduce_by_key(mapped_pairs)

first_driver = min(metrics_by_driver, key=int)
first_driver, metrics_by_driver[first_driver]

('10', {'total_hours': 3232.0, 'total_miles': 147150.0, 'weeks': 52})

In [6]:
# El maestro transforma cada conductor en una pareja de la misma clave y su identidad legible.

def map_driver_names(records):
    names_by_driver = {}

    for record in records:
        names_by_driver[record["driverId"]] = record["name"]

    return names_by_driver

In [7]:
# La segunda fuente queda indexada por la misma clave que produjeron las métricas.

names_by_driver = map_driver_names(drivers)

first_driver, names_by_driver[first_driver]

('10', 'George Vetticaden')

In [8]:
# El join conserva los conductores presentes en ambas fuentes y combina sus atributos.

def join_by_key(metrics, names):
    activity = []

    for driver_id, metric in metrics.items():
        if driver_id in names:
            activity.append({"driverId": driver_id, "name": names[driver_id], **metric})

    return sorted(activity, key=lambda record: record["total_miles"], reverse=True)

In [9]:
# La respuesta final combina la identidad del conductor con sus medidas acumuladas.

driver_activity = join_by_key(metrics_by_driver, names_by_driver)

with JOIN_OUTPUT.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(driver_activity[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(driver_activity)

with RESULT.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(driver_activity[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(driver_activity)

driver_activity[:10]

[{'driverId': '11',
  'name': 'Jamie Engesser',
  'total_hours': 3642.0,
  'total_miles': 179300.0,
  'weeks': 52},
 {'driverId': '10',
  'name': 'George Vetticaden',
  'total_hours': 3232.0,
  'total_miles': 147150.0,
  'weeks': 52},
 {'driverId': '33',
  'name': 'Sridhara Sabbella',
  'total_hours': 2759.0,
  'total_miles': 139285.0,
  'weeks': 52},
 {'driverId': '25',
  'name': 'Jean-Philippe Playe',
  'total_hours': 2723.0,
  'total_miles': 139180.0,
  'weeks': 52},
 {'driverId': '39',
  'name': 'David Kaiser',
  'total_hours': 2745.0,
  'total_miles': 138788.0,
  'weeks': 52},
 {'driverId': '15',
  'name': 'Rohit Bakshi',
  'total_hours': 2734.0,
  'total_miles': 138750.0,
  'weeks': 52},
 {'driverId': '35',
  'name': 'Emil Siemes',
  'total_hours': 2728.0,
  'total_miles': 138727.0,
  'weeks': 52},
 {'driverId': '21',
  'name': 'Jeff Markham',
  'total_hours': 2751.0,
  'total_miles': 138719.0,
  'weeks': 52},
 {'driverId': '41',
  'name': 'Greg Phillips',
  'total_hours': 2723.0